# 2.1 — Procesamiento: calor extremo diario (Open-Meteo)

**Capa 2 del protocolo (clima, complemento diario).** El SMN solo da promedios mensuales, y con eso no se pueden detectar olas de calor. Aquí se usa la serie **diaria** de Open-Meteo para marcar olas de calor y resumirlas **por mes** (para el panel) y **por semana** (para el análisis de rezagos).

Metodología: `docs/docs/protocolo_eda.md` (sección 2.1) y `docs/docs/heat_index.md`.

| | |
|---|---|
| **Entrada** | `data/raw/openmeteo_hermosillo_daily_1991_2020.csv` (Tmax, Tmin y humedad diarias; se descarga si no existe) |
| **Salidas (opcionales)** | `data/interim/clima/openmeteo_hermosillo_mensual.csv` y `openmeteo_hermosillo_semanal.csv` |

## Checklist

- [x] Serie diaria completa y sin valores imposibles.
- [x] Umbrales P95 y olas de calor (fija, P95 y húmeda).
- [x] Agregación mensual y contraste con el SMN.
- [x] Agregación semanal y empate con las semanas de IRA.

### Ideas clave

- **Open-Meteo** es un *reanálisis*: un modelo que estima el clima de cada día para un punto del mapa (aquí, el centro de Hermosillo). No es una estación.
- Se usan datos de **1991 a 2020**: los primeros años solo sirven para saber qué es "calor normal"; el análisis usa **2015–2020**.
- **Percentil 95 (P95):** el valor que solo supera el 5 % de los días de ese mes.
- **Heat index:** la sensación térmica, que combina temperatura y humedad.
- **Semana epidemiológica:** de domingo a sábado; así reporta Salud los casos de IRA.

### Tres formas de definir una ola de calor

En las tres, una ola son **3 o más días seguidos** que cumplen la condición:

| Columna | Condición de cada día |
|---------|-----------------------|
| `en_ola` (fija) | Tmax ≥ 35 °C |
| `en_ola_p95` (P95) | Tmax **y** Tmin por encima del P95 de su mes |
| `en_ola_humedo` (húmeda) | Heat index ≥ 40 °C |

### Funciones de `panel.py` que se usan

| Función | Qué hace |
|---------|----------|
| `load_openmeteo_daily()` | Lee (o descarga) la serie diaria |
| `_openmeteo_daily_flags()` | Agrega a cada día los umbrales P95, el heat index y las columnas `en_ola*` |
| `openmeteo_hermosillo_monthly()` | Resume por mes (**es lo que entra al panel**) |
| `openmeteo_hermosillo_weekly()` | Resume por semana epidemiológica |
| `climate_hermosillo()` | Serie mensual del SMN (notebook 2.0), para comparar |
| `health_ira_weekly()` | Casos de IRA por semana, para comprobar que las semanas coinciden |

## Configuración e imports

Se cargan las librerías, las constantes de `config.py` y las funciones de `panel.py`.

**Si falla** con `No module named 'sonora_health_climate_eda'`, el kernel elegido no es `.venv`.

In [ ]:
from pathlib import Path  # Path: herramienta para manejar rutas de carpetas y archivos
import sys  # sys: permite cambiar dónde busca módulos Python

# Este notebook vive en notebooks/02_procesamiento/. Con .parent.parent subimos
# dos carpetas hasta llegar a la raíz del proyecto.
ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
# Si la raíz del proyecto no está en la lista de búsqueda de Python, la agrega.
# Así se puede importar el paquete `sonora_health_climate_eda`.
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd  # pandas: trabajar con tablas (DataFrames)
import matplotlib.pyplot as plt  # matplotlib: hacer gráficas

# De config.py: constantes del proyecto
from sonora_health_climate_eda.config import (
    HEAT_INDEX_DANGER_C,  # 40.0 °C: umbral de heat index para la ola húmeda
    INTERIM_DATA_DIR,  # ruta de la carpeta data/interim
    PANEL_END_YEAR,  # último año del panel (2020)
    PANEL_START_YEAR,  # primer año del panel (2015)
)
# De panel.py: funciones que leen Open-Meteo y calculan las olas de calor
from sonora_health_climate_eda.panel import (
    _openmeteo_daily_flags,  # privada: expone los flags diarios que usa el panel
    climate_hermosillo,  # serie mensual SMN (notebook 2.0), para comparar
    health_ira_weekly,  # casos de IRA por semana, para comprobar que las semanas empatan
    load_openmeteo_daily,  # lee (o descarga) la serie diaria de Open-Meteo
    openmeteo_hermosillo_monthly,  # pasa la serie diaria a métricas mensuales
    openmeteo_hermosillo_weekly,  # pasa la serie diaria a métricas por semana epidemiológica
)

# Carpeta donde se guardaría el resultado: data/interim/clima
OUT_DIR = INTERIM_DATA_DIR / "clima"
# Las tres variables diarias que vamos a revisar
VARS_DIARIAS = ["tmax_c", "tmin_c", "humedad_rel_pct"]

print(f"Ventana del panel: {PANEL_START_YEAR} a {PANEL_END_YEAR} | umbral heat index: {HEAT_INDEX_DANGER_C:.0f} °C")

## Completitud de la serie diaria

Se revisa que no falten días, que no haya fechas repetidas ni valores vacíos. Importa porque una ola se busca como **días seguidos**: un día faltante la partiría.

**Resultado:** 10,958 días (1991–2020), sin faltantes, duplicados ni vacíos.

In [ ]:
# Lee la serie diaria. La primera vez la descarga de internet y la guarda en
# data/raw/; las siguientes veces solo lee ese CSV.
diario = load_openmeteo_daily()
# len() = número de filas (días). :, pone comas en los miles; :%Y-%m-%d da formato a la fecha.
print(f"{len(diario):,} días, {diario['fecha'].min():%Y-%m-%d} a {diario['fecha'].max():%Y-%m-%d}")

# Calendario completo: todos los días (freq="D") entre la primera y la última fecha
calendario = pd.date_range(diario["fecha"].min(), diario["fecha"].max(), freq="D")
# .difference() = fechas que están en el calendario pero NO en la tabla (días faltantes)
faltan = calendario.difference(diario["fecha"])
# .duplicated() marca True las fechas repetidas; .sum() las cuenta
print(f"Días faltantes: {len(faltan)} | fechas duplicadas: {diario['fecha'].duplicated().sum()}")
print("\nNaN por variable:")
# isna() marca los vacíos y sum() los cuenta por columna
print(diario[VARS_DIARIAS].isna().sum())

## Rangos físicos

Se buscan días imposibles: mínima mayor que la máxima, o humedad fuera de 0–100 %.

**Resultado:** ninguno. Tmax va de 8.5 a 47.1 °C y Tmin de -0.6 a 31.1 °C, valores normales para Hermosillo.

In [ ]:
# Filtra los días en los que la mínima es mayor que la máxima (imposible)
tmin_mayor = diario[diario["tmin_c"] > diario["tmax_c"]]
# Filtra los días con humedad menor que 0 O mayor que 100 (| significa "o")
hr_fuera = diario[(diario["humedad_rel_pct"] < 0) | (diario["humedad_rel_pct"] > 100)]
print(f"Días con Tmin > Tmax: {len(tmin_mayor)} | HR fuera de [0, 100]: {len(hr_fuera)}")
# describe() resume cada columna; .T voltea la tabla para que cada variable sea una fila
diario[VARS_DIARIAS].describe().T

## Umbrales P95 por mes

Para cada mes se calcula qué temperatura solo supera el 5 % de los días (con todos los años 1991–2020). Así, 35 °C cuenta como extremo en marzo pero no en julio.

**Resultado:** el umbral de Tmax va de 28.9 °C (diciembre) a 43.1 °C (junio).

In [ ]:
# Tabla diaria con umbrales P95, heat index y flags de ola de calor (1991–2020)
flags = _openmeteo_daily_flags()
umbrales = (
    flags.groupby(flags["fecha"].dt.month)[["thr_tmax_p95", "thr_tmin_p95"]]  # agrupa por mes calendario (1–12)
    .first()  # todos los días de un mes tienen el mismo umbral: basta con el primero
    .rename_axis("mes")  # el índice se llamará "mes" en lugar de "fecha"
    .round(1)  # redondea a 1 decimal
)
umbrales  # al ser la última línea, la celda muestra la tabla

## Días en ola de calor (2015–2020)

Se cuentan los días que caen dentro de una ola con cada definición, y se grafica en qué meses ocurren.

**Resultado:**
- **Fija:** 774 días; cubre casi todo el verano, así que distingue poco.
- **P95:** solo 21 días; marca calor raro para la época.
- **Húmeda:** 692 días; cubre casi todo julio y agosto.

In [ ]:
# Diccionario: nombre de la columna -> etiqueta legible para tablas y gráficas
DEF_OLAS = {
    "en_ola": "fija ≥35 °C",
    "en_ola_p95": "P95 adaptado",
    "en_ola_humedo": f"heat index ≥{HEAT_INDEX_DANGER_C:.0f} °C",
}
# Se queda solo con los días de 2015 a 2020. between() incluye ambos extremos.
ventana = flags[flags["fecha"].dt.year.between(PANEL_START_YEAR, PANEL_END_YEAR)]
print("Días dentro de una ola de calor:")
# Las columnas en_ola* valen 1 o 0, así que sum() cuenta los días en ola.
# list(DEF_OLAS) da las llaves del diccionario; .rename() cambia los nombres por las etiquetas.
print(ventana[list(DEF_OLAS)].sum().rename(DEF_OLAS))

# Por mes calendario: el promedio de una columna de 1 y 0 es la fracción de días en ola
frac_mes = ventana.groupby(ventana["fecha"].dt.month)[list(DEF_OLAS)].mean().rename(columns=DEF_OLAS)
ax = frac_mes.plot(kind="bar", figsize=(10, 4))  # gráfica de barras, una barra por definición
ax.set_xlabel("Mes calendario")  # etiqueta del eje horizontal
ax.set_ylabel("Fracción de días")  # etiqueta del eje vertical
ax.set_title("Fracción de días en ola de calor por mes calendario")  # título
plt.tight_layout()  # ajusta los márgenes para que no se corte nada
plt.show()  # muestra la gráfica

### Verificación del heat index

Se revisan los valores más altos del heat index.

**Resultado:** llega a 70.8 °C, un valor irreal. La fórmula combina la temperatura **máxima** con la humedad **promedio** del día, y eso exagera el resultado. Sirve para comparar días entre sí, pero no como valor exacto (anotado en las decisiones).

In [ ]:
# Resumen estadístico del heat index en la ventana del panel
print(ventana["heat_index_c"].describe().round(1))
# Cuenta los días con heat index >= 54 °C (categoría "peligro extremo" del NWS)
print(f"\nDías con heat index ≥54 °C: {(ventana['heat_index_c'] >= 54).sum()}")
# Fracción de días con humedad media < 40 % (donde Rothfusz necesita ajustes)
print(f"Fracción de días con HR media < 40 %: {(ventana['humedad_rel_pct'] < 40).mean():.0%}")
# Los 5 días con heat index más alto, con su Tmax y su humedad (redondea solo el heat index)
ventana.nlargest(5, "heat_index_c")[["fecha", "tmax_c", "humedad_rel_pct", "heat_index_c"]].round({"heat_index_c": 1})

## Agregación mensual

`openmeteo_hermosillo_monthly()` resume los días en una fila por mes: temperaturas, humedad, días de calor y **número de olas** (`n_olas_*`). Cada ola se cuenta una vez, en el mes en que **empieza**.

**Resultado:** 72 meses. En 2015–2020 hubo 69 olas fijas, 37 húmedas y solo 5 P95 (marzo y octubre de 2017, agosto de 2019 y dos en noviembre de 2020).

In [ ]:
# Métricas mensuales de calor (lo que entra al panel), de 1991 a 2020
om_mensual = openmeteo_hermosillo_monthly()
# Se queda solo con los meses de 2015 a 2020
om_mensual = om_mensual[om_mensual["fecha_mes"].dt.year.between(PANEL_START_YEAR, PANEL_END_YEAR)]
print(f"{len(om_mensual)} meses en la ventana del panel")

# Columnas con el número de olas que empiezan en cada mes
COLS_N_OLAS = ["n_olas_calor", "n_olas_calor_p95", "n_olas_calor_humedo"]
print("\nNúmero de olas por año:")
# Agrupa por año y suma las olas de los 12 meses
print(om_mensual.groupby(om_mensual["fecha_mes"].dt.year)[COLS_N_OLAS].sum())

print("\nMeses con al menos una ola P95:")
# Filtra los meses con n_olas_calor_p95 > 0 y muestra solo tres columnas
print(om_mensual.loc[om_mensual["n_olas_calor_p95"] > 0, ["fecha_mes", "n_olas_calor_p95", "dias_ola_calor_p95"]])

om_mensual.head()  # muestra las primeras 5 filas

## Contraste Open-Meteo vs. SMN

Se compara la Tmax media mensual de las dos fuentes. Lo importante es que se muevan juntas, aunque una marque más que la otra.

**Resultado:** se mueven casi igual (correlación 0.994), pero Open-Meteo marca unos **2.2 °C menos** que las estaciones del SMN.

In [ ]:
# Tmax media mensual del SMN (la serie del notebook 2.0); solo dos columnas
smn = climate_hermosillo()[["fecha_mes", "temp_max_media_c"]]
# Une Open-Meteo y SMN por mes. how="inner" = solo meses presentes en ambas.
# dropna() quita los meses con algún vacío (los 3 diciembres dañados del SMN).
comp = om_mensual[["fecha_mes", "tmax_media_mes"]].merge(smn, on="fecha_mes", how="inner").dropna()
print(f"Meses comparables: {len(comp)}")
# .corr() = correlación de Pearson entre las dos columnas
print(f"Correlación: {comp['tmax_media_mes'].corr(comp['temp_max_media_c']):.3f}")
# Diferencia mes a mes y su promedio; :+.2f muestra el signo (+ o −) y 2 decimales
print(f"Sesgo medio (Open-Meteo − SMN): {(comp['tmax_media_mes'] - comp['temp_max_media_c']).mean():+.2f} °C")

# Diagrama de dispersión: eje X = SMN, eje Y = Open-Meteo, un punto por mes
ax = comp.plot.scatter(x="temp_max_media_c", y="tmax_media_mes", figsize=(5, 5))
# Límites de la línea de referencia: del valor más bajo al más alto de ambas columnas
lims = [comp[["temp_max_media_c", "tmax_media_mes"]].min().min(), comp[["temp_max_media_c", "tmax_media_mes"]].max().max()]
ax.plot(lims, lims, "--", color="gray")  # línea gris punteada: "ambas fuentes iguales"
ax.set_xlabel("SMN — Tmax media mensual (°C)")
ax.set_ylabel("Open-Meteo — Tmax media mensual (°C)")
plt.tight_layout()
plt.show()

## Agregación semanal

`openmeteo_hermosillo_weekly()` hace lo mismo que la mensual, pero por **semana epidemiológica** (de domingo a sábado). Sirve para estudiar si el calor de una semana se refleja en los casos de IRA de las semanas siguientes.

**Resultado:** 313 semanas, todas empiezan en domingo y el total de olas es igual al mensual. Solo la última semana está incompleta (5 días), y no afecta porque es invierno.

In [ ]:
# Métricas de calor por semana epidemiológica, de 1991 a 2020
om_semanal = openmeteo_hermosillo_weekly()
# Se queda solo con las semanas que empiezan entre 2015 y 2020
om_semanal = om_semanal[om_semanal["fecha_semana"].dt.year.between(PANEL_START_YEAR, PANEL_END_YEAR)]
print(f"{len(om_semanal)} semanas, de {om_semanal['fecha_semana'].min():%Y-%m-%d} a {om_semanal['fecha_semana'].max():%Y-%m-%d}")
# day_name() da el nombre del día; unique() deja los valores sin repetir (debe ser solo Sunday)
print("Días de la semana en fecha_semana:", list(om_semanal["fecha_semana"].dt.day_name().unique()))

# Semanas incompletas: se cuentan los días de datos que caen en cada semana.
# Se repite la misma cuenta que hace la función para ubicar el domingo de cada día.
dias_desde_domingo = (diario["fecha"].dt.weekday + 1) % 7  # domingo=0, lunes=1 … sábado=6
semana_de_cada_dia = diario["fecha"] - pd.to_timedelta(dias_desde_domingo, unit="D")  # domingo de su semana
dias_por_semana = semana_de_cada_dia.value_counts()  # cuántos días tiene cada semana
# Solo las semanas de la ventana del panel con menos de 7 días
en_ventana = dias_por_semana.index.year.isin(range(PANEL_START_YEAR, PANEL_END_YEAR + 1))
incompletas = dias_por_semana[(dias_por_semana < 7) & en_ventana]
print(f"\nSemanas incompletas en la ventana: {len(incompletas)}")
print(incompletas)

# Total de olas: debe coincidir con la versión mensual (COLS_N_OLAS viene de la celda mensual)
print("\nTotal de olas, semanal vs. mensual:")
print(pd.DataFrame({"semanal": om_semanal[COLS_N_OLAS].sum(), "mensual": om_mensual[COLS_N_OLAS].sum()}))

### ¿Las semanas coinciden con las de IRA?

Se unen las semanas de Open-Meteo con las de casos de IRA. Si todas salen como `both`, se pueden juntar en un panel semanal.

**Resultado:** las 313 semanas coinciden.

In [ ]:
# Casos de IRA de Hermosillo por semana epidemiológica (2015–2020)
ira_semanal = health_ira_weekly()
# Une por fecha_semana. how="outer" conserva todas las semanas de ambas tablas;
# indicator=True agrega la columna _merge: both / left_only / right_only
empate = om_semanal[["fecha_semana"]].merge(
    ira_semanal[["fecha_semana", "casos_ira"]], on="fecha_semana", how="outer", indicator=True
)
# Cuenta cuántas semanas hay en cada caso
empate["_merge"].value_counts()

## DataFrames finales

Primeras 5 filas de la tabla mensual (72 × 14) y de la semanal (313 × 14). Tienen las mismas métricas; cambia la fecha (`fecha_mes` / `fecha_semana`) y el sufijo (`_mes` / `_sem`).

**¿De dónde viene cada dato?** Estas tablas salen de **Open-Meteo** (olas, humedad, heat index). En el panel mensual se juntan con la temperatura mensual del **SMN** (notebook 2.0). La tabla semanal será la base del futuro panel semanal; ahí no entra el SMN porque solo existe por mes.

In [ ]:
# Ajuste de pandas para ver todas las columnas (por defecto oculta algunas con "...")
pd.set_option("display.max_columns", None)

# shape = (número de filas, número de columnas)
print(f"Mensual: {om_mensual.shape} | Semanal: {om_semanal.shape}")

# display() muestra una tabla con formato dentro de la celda (aquí hay dos tablas).
# head() = primeras 5 filas; round(1) redondea a 1 decimal para leer más fácil.
print("\nTabla MENSUAL (primeras 5 filas):")
display(om_mensual.head().round(1))
print("Tabla SEMANAL (primeras 5 filas):")
display(om_semanal.head().round(1))

## Guardar salida

Guarda las dos tablas en `data/interim/clima/` para revisarlas fuera del notebook (el panel no lee estos archivos). Está comentado a propósito: para guardar, quita los `#` de las líneas de código.

In [ ]:
# TODO: guardar una vez validadas las métricas
# Crea la carpeta data/interim/clima si no existe (parents=True crea también las intermedias)
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# Guarda la tabla en CSV; index=False evita guardar el número de fila como columna extra
# om_mensual.to_csv(OUT_DIR / "openmeteo_hermosillo_mensual.csv", index=False)
# om_semanal.to_csv(OUT_DIR / "openmeteo_hermosillo_semanal.csv", index=False)

## Decisiones de procesamiento

Si una decisión cambia el panel, se implementa en `sonora_health_climate_eda/panel.py`.

| Tema | Decisión | Por qué |
|------|----------|---------|
| Serie diaria 1991–2020 | Sin cambios | Completa y sin valores imposibles |
| Ola fija (≥35 °C) | Se conserva como referencia (`n_olas_calor`) | Es la convención del SMN, pero cubre casi todo el verano |
| Ola P95 | Métrica principal (`n_olas_calor_p95`) | Detecta calor fuera de temporada; ojo: solo 5 olas en 6 años |
| Heat index | Usar solo para comparar días. **Pendiente con el equipo:** usar la humedad mínima o subir el umbral | Exagera los valores (máximo de 70.8 °C) |
| Open-Meteo vs. SMN | Sin corrección | Se mueven juntas; Open-Meteo marca ~2.2 °C menos |
| Agregación semanal | Se agrega sin cambiar la mensual | Las 313 semanas coinciden con las de IRA |
| Última semana (27-dic-2020) | Sin cambios | Solo 5 días, pero es invierno y no tiene olas |